# Day 28: BPE Tokenizer from Scratch

**Layer:** Implementation | **Prerequisite:** Day 01 (Tokenization)


## Concept Overview

Byte Pair Encoding (BPE) learns a vocabulary of subword tokens by iteratively merging the most frequent pair of symbols. Starting from characters, each merge step creates a new token. The final vocabulary is used to tokenize new text by applying the merge rules in order.


In [ ]:
import collections, re, json
import numpy as np
import matplotlib.pyplot as plt

print('BPE tokenizer implementation')


## 1. BPE Training


In [ ]:
def get_vocab(text):
    """Initialize vocab from character-level tokenization."""
    vocab = collections.defaultdict(int)
    for word in text.split():
        chars = ' '.join(list(word)) + ' </w>'
        vocab[chars] += 1
    return vocab

def get_stats(vocab):
    """Count frequency of each symbol pair."""
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    """Merge the most frequent pair in all vocab entries."""
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    new_vocab = {}
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab

def train_bpe(text, num_merges=50):
    vocab = get_vocab(text)
    merges = []
    for i in range(num_merges):
        pairs = get_stats(vocab)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        vocab = merge_vocab(best, vocab)
        merges.append(best)
    return vocab, merges

text = 'low lower newest widest low lower'
vocab, merges = train_bpe(text, num_merges=10)
print('Training text:', text)
print('\nFirst 10 BPE merges:')
for i, merge in enumerate(merges):
    print(f'  {i+1}: {merge[0]} + {merge[1]} -> {merge[0]+merge[1]}')
print('\nFinal vocabulary:')
for word, freq in sorted(vocab.items(), key=lambda x: -x[1]):
    print(f'  {word!r}: {freq}')


## 2. BPE Tokenization (Encoding)


In [ ]:
def tokenize(word, merges):
    """Apply BPE merges to tokenize a word."""
    chars = list(word) + ['</w>']
    tokens = chars
    merge_set = {pair: i for i, pair in enumerate(merges)}
    
    while True:
        pairs = [(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]
        pair_priorities = {p: merge_set[p] for p in pairs if p in merge_set}
        if not pair_priorities:
            break
        best = min(pair_priorities, key=pair_priorities.get)
        i = 0
        new_tokens = []
        while i < len(tokens):
            if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == best:
                new_tokens.append(tokens[i] + tokens[i+1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens

# Test
for word in ['lower', 'lowest', 'newer', 'widest']:
    tokens = tokenize(word, merges)
    print(f'{word!r:12} -> {tokens}')


## 3. Compression Analysis


In [ ]:
corpus = 'inference engineering is the discipline of serving generative AI models in production faster cheaper and more reliably'
vocab, merges = train_bpe(corpus, num_merges=30)

# Compare character-level vs BPE token count
char_tokens = len(corpus.replace(' ', ''))
bpe_tokens = sum(len(tokenize(w, merges)) for w in corpus.split())
print(f'Corpus: {len(corpus.split())} words')
print(f'Character tokens: {char_tokens}')
print(f'BPE tokens (30 merges): {bpe_tokens}')
print(f'Compression: {char_tokens/bpe_tokens:.2f}x')

# Plot compression vs merge count
counts = []
for n in range(0, 35, 5):
    v, m = train_bpe(corpus, num_merges=n)
    total = sum(len(tokenize(w, m)) for w in corpus.split())
    counts.append(total)

plt.plot(range(0, 35, 5), counts, 'b-o')
plt.xlabel('Number of BPE merges'); plt.ylabel('Token count')
plt.title('BPE Compression vs Number of Merges')
plt.grid(True)
plt.savefig('bpe_compression.png', dpi=100, bbox_inches='tight')
plt.show()


## Experiments: Try These

1. Scale to a larger corpus. How many merges does it take to reach vocabulary size 50K?
2. Implement a regex pre-tokenizer (like GPT-2's) that splits on whitespace/punctuation before BPE.
3. Compare BPE vocabulary to a character-level vocabulary on out-of-distribution text.


## Key Takeaways

- BPE greedily merges the most frequent symbol pair in each training step — this is an O(V² T) algorithm for vocabulary size V and training size T.
- The merge rules form a priority queue — tokenizing new text applies rules in training order, greedily merging the highest-priority pair first.
- BPE balances vocabulary size against tokenization granularity — 32K-100K merges is typical for LLMs (GPT-2: 50K, Llama-3: 128K).
- The `</w>` end-of-word marker distinguishes subwords that occur at word boundaries vs in the middle.

## References
- Sennrich et al. (2016), "Neural Machine Translation of Rare Words with Subword Units"
- OpenAI tiktoken repository
